# Chapter 3 · Variables and Measurement Scales: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, one synthetic day of the triage log of an adult emergency department, through the clinical team's questions; only the sections with a calculation appear (3.1 and 3.7). **Part B** gives the written solution of every chapter exercise (1 to 13; statements are in the book) and, for the exercises whose solution needs a calculation (12 and 13), its code and result. The code is plain Python 3 with its standard library only. Run the cells in order.

## Part A. The case

**Data.** `data/triage.csv`: 200 rows, one per visit of March 9, 2026, and 12 columns: `visit_id`, `arrived_at`, `seen_at` (empty when the patient left before being seen), `arrival_route` (1 walk-in, 2 ambulance, 3 transfer), `sex`, `age_years`, `triage_level` (1 most urgent to 5 least urgent), `pain` (none, mild, moderate, severe), `temp_c` (empty when not recorded), `heart_rate`, `prior_visits`, and `admitted`. `data/triage_check.csv`: the first 12 visits as they were first typed, with three recording problems planted for the validation step. Each cell carries the number of the book section it belongs to.

### 3.1 Introduction: the data

In [1]:
# 1. Read the log: every cell arrives as text, the age included
import csv
from datetime import datetime

with open("data/triage.csv", newline="") as stream:
    visits = list(csv.DictReader(stream))
print("The log has", len(visits), "records (one per visit) and", len(visits[0]), "fields.")
print("Type of age_years as read:", type(visits[0]["age_years"]).__name__)

# 2. The identifier distinguishes the records and finds any of them; it measures nothing
by_id = {}
for row in visits:
    by_id[row["visit_id"]] = row
print("Distinct visit_id:", len(by_id))

The log has 200 records (one per visit) and 12 fields.
Type of age_years as read: str
Distinct visit_id: 200


### 3.7 The triage log and its dictionary

*What does the report's average of codes mean, and how is each variable's scale recorded so that a program can apply it?*

In [2]:
# 3. The report's average: what the system did with the arrival route codes (two numberings of the same categories)
route_a = {"1": 1, "2": 2, "3": 3}
route_b = {"1": 3, "2": 1, "3": 2}
total_a = 0
total_b = 0
for row in visits:
    total_a = total_a + route_a[row["arrival_route"]]
    total_b = total_b + route_b[row["arrival_route"]]
print("Average route with the numbering 1-2-3:", total_a / len(visits))
print("Average route with the numbering 3-1-2:", total_b / len(visits))

# 4. The same for triage level, which is ordinal: the order is kept, the spacing changes
level_a = {"1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
level_b = {"1": 1, "2": 2, "3": 4, "4": 8, "5": 16}
total_a = 0
total_b = 0
for row in visits:
    total_a = total_a + level_a[row["triage_level"]]
    total_b = total_b + level_b[row["triage_level"]]
print("Average level with the spacing 1-2-3-4-5:", total_a / len(visits))
print("Average level with the spacing 1-2-4-8-16:", total_b / len(visits))

Average route with the numbering 1-2-3: 1.29
Average route with the numbering 3-1-2: 2.615
Average level with the spacing 1-2-3-4-5: 3.345
Average level with the spacing 1-2-4-8-16: 6.035


In [3]:
# 5. Derived variables: time to be seen (minutes) and fever; a visit with no time seen left before being seen
for row in visits:
    row["wait_min"] = None
    if row["seen_at"] != "":
        arrived = datetime.fromisoformat(row["arrived_at"])
        seen = datetime.fromisoformat(row["seen_at"])
        row["wait_min"] = (seen - arrived).total_seconds() / 60
    row["fever"] = None
    if row["temp_c"] != "":
        if float(row["temp_c"]) >= 38.0:
            row["fever"] = "yes"
        else:
            row["fever"] = "no"
print("Time to be seen, V0001:", by_id["V0001"]["wait_min"], "minutes; V0002:", by_id["V0002"]["wait_min"], "minutes")
left = []
for row in visits:
    if row["wait_min"] is None:
        left.append(row["visit_id"])
print("Left before being seen:", len(left), "visits", left)
for visit_id in ["V0074", "V0026", "V0174"]:
    print("Visit", visit_id, "- temperature", by_id[visit_id]["temp_c"], "- fever:", by_id[visit_id]["fever"])

Time to be seen, V0001: 87.0 minutes; V0002: 25.0 minutes
Left before being seen: 3 visits ['V0025', 'V0050', 'V0093']
Visit V0074 - temperature 37.6 - fever: no
Visit V0026 - temperature 38.2 - fever: yes
Visit V0174 - temperature 40.0 - fever: yes


In [4]:
# 6. The scale of each variable decides the summary: counts for categories
SCALE = {"visit_id": "identifier", "arrival_route": "nominal", "sex": "nominal",
         "admitted": "nominal", "fever": "nominal", "triage_level": "ordinal",
         "pain": "ordinal", "arrived_at": "interval", "seen_at": "interval",
         "temp_c": "interval", "age_years": "ratio", "heart_rate": "ratio",
         "prior_visits": "ratio", "wait_min": "ratio"}
for scale in ["identifier", "nominal", "ordinal", "interval", "ratio"]:
    names = []
    for name in SCALE:
        if SCALE[name] == scale:
            names.append(name)
    print(scale + ":", ", ".join(names))

# The counts of the nominal and ordinal variables, in the order of the scale (missing values are counted apart)
ORDER = {"arrival_route": ["1", "2", "3"], "sex": ["F", "M"],
         "admitted": ["yes", "no"], "fever": ["yes", "no"],
         "triage_level": ["1", "2", "3", "4", "5"],
         "pain": ["none", "mild", "moderate", "severe"]}
for variable in SCALE:
    if SCALE[variable] in ("nominal", "ordinal"):
        counts = {}
        for value in ORDER[variable]:
            counts[value] = 0
        counts["missing"] = 0
        for row in visits:
            value = row[variable]
            if value is None or value == "":
                value = "missing"
            counts[value] = counts[value] + 1
        shown = {}
        for value in counts:
            if counts[value] > 0:
                shown[value] = counts[value]
        print("Counts of", variable, "(" + SCALE[variable] + "):", shown)

identifier: visit_id
nominal: arrival_route, sex, admitted, fever
ordinal: triage_level, pain
interval: arrived_at, seen_at, temp_c
ratio: age_years, heart_rate, prior_visits, wait_min
Counts of arrival_route (nominal): {'1': 155, '2': 32, '3': 13}
Counts of sex (nominal): {'F': 102, 'M': 98}
Counts of admitted (nominal): {'yes': 54, 'no': 146}
Counts of fever (nominal): {'yes': 44, 'no': 152, 'missing': 4}
Counts of triage_level (ordinal): {'1': 5, '2': 23, '3': 85, '4': 72, '5': 15}
Counts of pain (ordinal): {'none': 38, 'mild': 80, 'moderate': 51, 'severe': 31}


In [5]:
# 7. Interval and ratio: the same three temperatures in three units, and the same two ages in two units
celsius = [37.0, 38.0, 39.0]
fahrenheit = []
kelvin = []
for c in celsius:
    fahrenheit.append(round(1.8 * c + 32, 1))
    kelvin.append(round(c + 273.15, 2))
print("Celsius", celsius, "- ratio of the last to the first:", round(celsius[2] / celsius[0], 3))
print("Fahrenheit", fahrenheit, "- ratio of the last to the first:", round(fahrenheit[2] / fahrenheit[0], 3))
print("Kelvin", kelvin, "- ratio of the last to the first:", round(kelvin[2] / kelvin[0], 3))
print("Age ratio, 80 years against 40 years:", 80 / 40, "; in months:", (80 * 12) / (40 * 12))

Celsius [37.0, 38.0, 39.0] - ratio of the last to the first: 1.054
Fahrenheit [98.6, 100.4, 102.2] - ratio of the last to the first: 1.037
Kelvin [310.15, 311.15, 312.15] - ratio of the last to the first: 1.006
Age ratio, 80 years against 40 years: 2.0 ; in months: 2.0


In [6]:
# 8. The dictionary's rules, applied to a small file typed by hand
rules = {
    "age_years": {"convert": int, "low": 18, "high": 110},
    "triage_level": {"convert": int, "low": 1, "high": 5},
    "temp_c": {"convert": float, "low": 30.0, "high": 43.0},
}
with open("data/triage_check.csv", newline="") as stream:
    check = list(csv.DictReader(stream))

validated = []
issues = []
for row in check:
    clean = {"visit_id": row["visit_id"]}
    for name in rules:
        rule = rules[name]
        text = row[name]
        clean[name] = None
        if text == "":
            print(row["visit_id"], name, "missing")
        else:
            value = rule["convert"](text)
            if rule["low"] <= value <= rule["high"]:
                clean[name] = value
            else:
                issues.append((row["visit_id"], name, text))
                print(row["visit_id"], name, "not allowed:", value)
    validated.append(clean)
print("For review:", issues)
checked = {}
for record in validated:
    checked[record["visit_id"]] = record
print("V0004 triage level after validation:", checked["V0004"]["triage_level"])

V0004 triage_level not allowed: 6
V0007 temp_c missing
V0008 age_years not allowed: 430
V0010 temp_c missing
For review: [('V0004', 'triage_level', '6'), ('V0008', 'age_years', '430')]
V0004 triage level after validation: None


In [7]:
# 9. The summaries that the next section reports
n = len(visits)
urgent = 0
ambulance = 0
admitted = 0
fever_yes = 0
fever_no = 0
for row in visits:
    if row["triage_level"] == "1" or row["triage_level"] == "2":
        urgent = urgent + 1
    if row["arrival_route"] == "2":
        ambulance = ambulance + 1
    if row["admitted"] == "yes":
        admitted = admitted + 1
    if row["fever"] == "yes":
        fever_yes = fever_yes + 1
    if row["fever"] == "no":
        fever_no = fever_no + 1
known = fever_yes + fever_no
print("Triage levels 1 and 2:", urgent, "of", n, "visits =", round(100 * urgent / n, 1), "%")
print("Direct ambulance:", ambulance, "of", n, "visits =", round(100 * ambulance / n, 1), "%")
print("Admitted:", admitted, "of", n, "visits =", round(100 * admitted / n, 1), "%")
print("Fever:", fever_yes, "of", known, "visits with a recorded temperature =", round(100 * fever_yes / known, 1), "%")

Triage levels 1 and 2: 28 of 200 visits = 14.0 %
Direct ambulance: 32 of 200 visits = 16.0 %
Admitted: 54 of 200 visits = 27.0 %
Fever: 44 of 196 visits with a recorded temperature = 22.4 %


## Part B. Exercise solutions

### 3.9 Exercise 1. A heart failure register

The unit of observation is one enrolled patient; each row is one patient. The
variables are sex, functional class, ejection fraction, number of admissions, and
date of enrollment. Record is an identifier: it distinguishes the rows and
measures nothing. Clinic has the value Cardiology in all four rows, so it is a
constant in this dataset and cannot distinguish one patient from another.

### 3.9 Exercise 2. Five columns in a laboratory

The accession number is an identifier. Specimen type, degree of hemolysis, and
rejection are qualitative variables: their values are categories. Potassium
concentration is a quantitative variable: it results from a measurement.

### 3.9 Exercise 3. Counting specimen types

The denominator is the 500 specimens received that week. Blood accounts for
$320/500=64\%$, urine for $140/500=28\%$, and other specimens for $40/500=8\%$.
The three proportions add to 100%. Specimen type is nominal, so the codes 1, 2,
and 3 are names. Their average, $(320\times1+140\times2+40\times3)/500=1.44$,
would change if the codes were reassigned while the specimens stayed the same.
It describes the numbering chosen by the system.

### 3.9 Exercise 4. Functional class in a cohort

Functional class is qualitative on an ordinal scale: the four classes have an
order, and the distances between them are not defined. Class III or IV comprises
$45+15=60$ patients, or $60/150=40\%$. The scale does not allow the statement that
a patient in class IV is twice as limited as a patient in class II, nor that the
step from I to II equals the step from III to IV.

### 3.9 Exercise 5. Two codings of pain

With the codes 0, 1, 2, 3, ward A gives $(0+0+3+3)/4=1.5$ and ward B gives
$(1+2+2+2)/4=1.75$, so ward B appears to have more pain. With the codes 0, 1, 2,
10, ward A gives $(0+0+10+10)/4=5.0$ and ward B again gives 1.75, so ward A
appears to have more pain. Both codings respect the order of the categories, and
the comparison reverses. Counts and proportions do not depend on the coding: in
ward A, two of four patients had severe pain and two had none; in ward B, no
patient had severe pain and all four had some pain.

### 3.9 Exercise 6. Meaningful and meaningless statements

(a) Meaningful: concentration is on a ratio scale, and zero means no potassium.
(b) Not meaningful: degrees Celsius form an interval scale with an arbitrary zero.
(c) Meaningful: dates are on an interval scale, and the statement is a difference.
(d) Meaningful: a duration is on a ratio scale.
(e) Not meaningful: triage level is ordinal, and its digits are ordered labels.

### 3.9 Exercise 7. Differences and quotients in two units

In Celsius, the rise from the first to the third patient is $39.5-36.5=3.0$ and
from the first to the second $37.5-36.5=1.0$, a quotient of 3. In Fahrenheit the
values are 97.7, 99.5, and 103.1; the rises are 5.4 and 1.8, and their quotient
is again 3. The quotient of the third and first values is $39.5/36.5\approx1.082$
in Celsius and $103.1/97.7\approx1.055$ in Fahrenheit. This follows from the arbitrary zeros of the Celsius and Fahrenheit scales: ratios of differences are preserved, and ratios of values are not.

### 3.9 Exercise 8. Discrete or continuous

The number of medications and the number of falls are discrete: they result from
counting. Weight, systolic blood pressure, and serum creatinine are continuous:
they result from measuring, and a finer instrument would distinguish intermediate
values. A blood pressure recorded to the nearest 2 mmHg is a whole number by construction,
and the variable is continuous all the same. The recording rule limits the
stored values; it does not change the nature of the quantity.

### 3.9 Exercise 9. Body mass index and obesity

The height squared is $1.80^2=3.24$. The indices are $96.9/3.24\approx29.9$,
$97.5/3.24\approx30.1$, and $145.8/3.24=45.0$. By the clinic's definition the first
patient is not obese and the other two are. The first two patients are most alike
in the index, differing by 0.2, and they fall in different categories. The second
and third are alike in the category and differ by 14.9. The index is a derived
quantitative variable on a ratio scale. The obesity status is a derived binary
variable on a nominal scale.

### 3.9 Exercise 10. An age recorded in groups

The registry cannot supply the proportion aged 70 or older. The group 65 and older
contains those patients together with patients aged 65 to 69, and nothing in the
record separates them. Age in years, or the date of birth together with the date of the record, should have been
recorded; any grouping of ages in years could then have been derived. Categorizing runs from the
finer record to the coarser one and cannot be reversed.

### 3.9 Exercise 11. A data dictionary for the register

| Variable | Type | Scale | Values or unit |
|---|---|---|---|
| Record | identifier |  | one per patient |
| Clinic | qualitative | nominal | constant in this file (Cardiology in every row) |
| Sex | qualitative, binary | nominal | F, M |
| Class | qualitative | ordinal | I, II, III, IV |
| EF | quantitative, continuous | ratio | percent, 0 to 100 |
| Admissions | quantitative, discrete | ratio | 0, 1, 2, ... in the previous year |
| Enrolled | quantitative | interval | calendar date |

### 3.9 Exercise 12. Pain and prior visits in the triage log

The counts in the order of the scale are 38, 80, 51, and 31 for none, mild,
moderate, and severe. At least moderate pain was recorded in $51+31=82$ of 200
visits, or 41%. After conversion with `int`, 29 visits were made by
patients with three or more visits in the preceding 12 months, or 14.5%.

In [8]:
# 1. Count the pain categories in the order of the scale and the visits of patients with three or more prior visits
order = ["none", "mild", "moderate", "severe"]
pain_counts = {}
for category in order:
    pain_counts[category] = 0
frequent = 0
for row in visits:
    pain_counts[row["pain"]] = pain_counts[row["pain"]] + 1
    if int(row["prior_visits"]) >= 3:
        frequent = frequent + 1
print("Pain counts:", pain_counts)

# 2. The two proportions
at_least_moderate = pain_counts["moderate"] + pain_counts["severe"]
print("At least moderate pain:", at_least_moderate, "of", len(visits), "visits =", round(100 * at_least_moderate / len(visits), 1), "%")
print("Three or more visits in the preceding 12 months:", frequent, "of", len(visits), "visits =", round(100 * frequent / len(visits), 1), "%")

Pain counts: {'none': 38, 'mild': 80, 'moderate': 51, 'severe': 31}
At least moderate pain: 82 of 200 visits = 41.0 %
Three or more visits in the preceding 12 months: 29 of 200 visits = 14.5 %


### 3.9 Exercise 13. A rule for heart rate

For this exercise, choose 30 and 220 beats per minute as limits that trigger
review, not as the boundaries of possible heart rates. The rule is
`{"convert": int, "low": 30, "high": 220}`, applied record by record: each heart rate is converted
and compared with the two limits. With these limits, no
heart rate in the 200 records is missing and none is flagged. A value inside the
limits can still be wrong: 84 typed instead of 48 passes the range check. A value
outside them requires investigation rather than automatic correction or deletion.
Verification requires comparison with the source record. The limits are teaching choices,
wide enough that no plausible adult heart rate is flagged. The loop below applies the rule to the 200 records.

In [9]:
# 1. The rule for heart rate: limits that trigger review (plausibility limits, not the boundaries of possible heart rates)
rule = {"convert": int, "low": 30, "high": 220}

# 2. Apply it to the 200 records
missing = 0
flagged = []
for row in visits:
    text = row["heart_rate"]
    if text == "":
        missing = missing + 1
    else:
        value = rule["convert"](text)
        if not (rule["low"] <= value <= rule["high"]):
            flagged.append(row["visit_id"])
print("Missing heart rates:", missing)
print("Heart rates not allowed:", len(flagged))

Missing heart rates: 0
Heart rates not allowed: 0


---
*Synthetic data (`generate_data.py`, seed 3). Teaching values only; nothing here is an estimate for a real department.*